# 02 ICEEMDAN Decomposition and Hierarchical Ward-RLN Reconstruction

This notebook decomposes precious-metal futures and spot returns with ICEEMDAN, then reconstructs STC/MTC/LTC components with Ward hierarchical clustering on `[log(mean period), log(RLN/T)]` IMF features.


In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from PyEMD import EMD
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.signal import argrelextrema, find_peaks
from scipy.stats import spearmanr
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "precious_metals_daily_log_returns.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "02_iceemdan_hierarchical_ward"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RETURN_COLUMNS = [
    "gold_futures",
    "silver_futures",
    "platinum_futures",
    "palladium_futures",
    "gold_spot",
    "silver_spot",
    "platinum_spot",
    "palladium_spot",
]

PARAMS = {
    "trials": 100,
    "epsilon": 0.2,
    "seed": 20260428,
    "spline_kind": "cubic",
    "max_imf": 12,
    "hierarchical_clusters": 3,
    "hierarchical_linkage": "ward",
    "reconstruction_tolerance": 1e-8,
}

TEXT_COLOR = "#222222"
AXIS_COLOR = "#333333"
GRID_COLOR = "#D9D9D9"
SERIES_COLORS = {
    "original": "#222222",
    "emd_reconstructed": "#0072B2",
    "scale_reconstructed": "#D55E00",
    "emd_error": "#555555",
    "scale_error": "#CC79A7",
}
SCALE_COLORS = {"STC": "#D55E00", "MTC": "#0072B2", "LTC": "#009E73", "residue": "#666666"}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": AXIS_COLOR,
        "axes.linewidth": 0.8,
        "axes.grid": True,
        "grid.color": GRID_COLOR,
        "grid.alpha": 0.55,
        "grid.linewidth": 0.6,
        "grid.linestyle": "-",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titlelocation": "left",
        "axes.titleweight": "semibold",
        "axes.titlesize": 10.5,
        "axes.labelsize": 9.5,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "xtick.color": TEXT_COLOR,
        "ytick.color": TEXT_COLOR,
        "axes.labelcolor": TEXT_COLOR,
        "text.color": TEXT_COLOR,
        "font.family": "DejaVu Sans",
        "font.size": 9.5,
        "legend.fontsize": 8.8,
        "legend.frameon": False,
        "legend.handlelength": 2.4,
        "lines.solid_capstyle": "round",
    }
)


def format_time_axis(ax: plt.Axes, ylabel: str | None = None, zero_line: bool = False) -> None:
    ax.set_axisbelow(True)
    ax.margins(x=0.01)
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="both", width=0.7, length=3.0, color=AXIS_COLOR)
    if ylabel:
        ax.set_ylabel(ylabel)
    if zero_line:
        ax.axhline(0, color=AXIS_COLOR, linewidth=0.65, alpha=0.65)

print(f"Input: {DATA_PATH}")
print(f"Output directory: {OUTPUT_DIR}")


Input: e:\@PythonProject\@Precious\precious-metal\data\processed\precious_metals_daily_log_returns.csv
Output directory: e:\@PythonProject\@Precious\precious-metal\data\processed\02_iceemdan_hierarchical_ward


In [2]:
returns = pd.read_csv(DATA_PATH, parse_dates=["date"])

assert list(returns.columns) == ["date", *RETURN_COLUMNS], returns.columns.tolist()
assert len(returns) == 3100, len(returns)
assert returns["date"].min() == pd.Timestamp("2012-01-04"), returns["date"].min()
assert returns["date"].max() == pd.Timestamp("2026-03-31"), returns["date"].max()
assert returns["date"].is_monotonic_increasing
assert returns["date"].duplicated().sum() == 0
assert returns[RETURN_COLUMNS].isna().sum().sum() == 0
assert all(returns[col].var() > 0 for col in RETURN_COLUMNS)

year_counts = returns.assign(year=returns["date"].dt.year).groupby("year").size()
low_common_days_after_2017 = year_counts.loc[year_counts.index >= 2017].to_dict()

input_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
input_checks = {
    "input_file": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "input_sha256": input_sha256,
    "n_rows": int(len(returns)),
    "n_series": len(RETURN_COLUMNS),
    "date_start": returns["date"].min().strftime("%Y-%m-%d"),
    "date_end": returns["date"].max().strftime("%Y-%m-%d"),
    "duplicate_dates": int(returns["date"].duplicated().sum()),
    "missing_values": int(returns[RETURN_COLUMNS].isna().sum().sum()),
    "year_counts": {str(k): int(v) for k, v in year_counts.items()},
    "limitation_note": "Common trading days decline to about 198-201 observations per full year after 2017, so long-term component interpretation should be conservative.",
}

print(json.dumps(input_checks, ensure_ascii=False, indent=2))

{
  "input_file": "data\\processed\\precious_metals_daily_log_returns.csv",
  "input_sha256": "7edc035e4616d8e9efff34acbe3ed7085e5dda8e4b91e095d4a34c92917cd15f",
  "n_rows": 3100,
  "n_series": 8,
  "date_start": "2012-01-04",
  "date_end": "2026-03-31",
  "duplicate_dates": 0,
  "missing_values": 0,
  "year_counts": {
    "2012": 251,
    "2013": 252,
    "2014": 252,
    "2015": 252,
    "2016": 252,
    "2017": 197,
    "2018": 198,
    "2019": 198,
    "2020": 201,
    "2021": 200,
    "2022": 199,
    "2023": 198,
    "2024": 198,
    "2025": 198,
    "2026": 54
  },
  "limitation_note": "Common trading days decline to about 198-201 observations per full year after 2017, so long-term component interpretation should be conservative."
}


In [3]:
@dataclass(frozen=True)
class DecompositionResult:
    imfs: np.ndarray
    residue: np.ndarray
    elapsed_seconds: float


def count_extrema(values: np.ndarray) -> int:
    values = np.asarray(values, dtype=float)
    if values.size < 3:
        return 0
    maxima = argrelextrema(values, np.greater)[0]
    minima = argrelextrema(values, np.less)[0]
    return int(len(maxima) + len(minima))


def emd_imfs_and_residue(emd: EMD, values: np.ndarray, max_imf: int = -1) -> tuple[np.ndarray, np.ndarray]:
    values = np.asarray(values, dtype=float)
    emd.emd(values, max_imf=max_imf)
    imfs, residue = emd.get_imfs_and_residue()
    if imfs.size == 0:
        return np.empty((0, values.size)), values.copy()
    return np.asarray(imfs, dtype=float), np.asarray(residue, dtype=float)


def local_mean(emd: EMD, values: np.ndarray) -> np.ndarray:
    """ICEEMDAN M(.) operator: residue after extracting the first IMF by EMD."""
    imfs, residue = emd_imfs_and_residue(emd, values, max_imf=1)
    if imfs.shape[0] == 0:
        return np.asarray(values, dtype=float).copy()
    return residue


def normalize_rows(values: np.ndarray) -> np.ndarray:
    means = values.mean(axis=1, keepdims=True)
    stds = values.std(axis=1, keepdims=True)
    return np.divide(values - means, stds, out=np.zeros_like(values), where=stds > 0)


def normalize_imfs(imfs: np.ndarray) -> np.ndarray:
    if imfs.size == 0:
        return imfs
    stds = imfs.std(axis=1, keepdims=True)
    return np.divide(imfs, stds, out=np.zeros_like(imfs), where=stds > 0)


def iceemdan_decompose(
    values: np.ndarray,
    *,
    trials: int,
    epsilon: float,
    seed: int,
    spline_kind: str,
    max_imf: int,
) -> DecompositionResult:
    """Direct ICEEMDAN implementation using PyEMD.EMD only.

    This follows the iterative residue construction in Colominas et al. (2014):
    M(.) is estimated by the local mean from EMD, white-noise IMFs E_k(w_i) are
    precomputed, and each IMF is obtained as the difference of consecutive residues.
    """
    start = time.perf_counter()
    values = np.asarray(values, dtype=float)
    n_obs = values.size
    signal_std = float(values.std(ddof=0))
    if signal_std <= 0:
        raise ValueError("ICEEMDAN requires a non-constant input series.")

    rng = np.random.default_rng(seed)
    emd = EMD(spline_kind=spline_kind)

    noises = normalize_rows(rng.standard_normal((trials, n_obs)))
    noise_imfs: list[np.ndarray] = []
    for noise in noises:
        imfs, _ = emd_imfs_and_residue(emd, noise, max_imf=max_imf)
        noise_imfs.append(normalize_imfs(imfs))

    def kth_noise_imf(trial_idx: int, imf_idx: int) -> np.ndarray:
        imfs = noise_imfs[trial_idx]
        if imf_idx < imfs.shape[0]:
            return imfs[imf_idx]
        return np.zeros(n_obs)

    first_residues = [
        local_mean(emd, values + epsilon * signal_std * kth_noise_imf(trial_idx, 0))
        for trial_idx in range(trials)
    ]
    previous_residue = np.mean(first_residues, axis=0)
    modes = [values - previous_residue]

    for imf_idx in range(1, max_imf):
        if count_extrema(previous_residue) < 2:
            break
        beta = epsilon * float(previous_residue.std(ddof=0))
        next_residues = [
            local_mean(emd, previous_residue + beta * kth_noise_imf(trial_idx, imf_idx))
            for trial_idx in range(trials)
        ]
        next_residue = np.mean(next_residues, axis=0)
        mode = previous_residue - next_residue
        if np.allclose(mode, 0.0, atol=1e-12, rtol=0.0):
            break
        modes.append(mode)
        previous_residue = next_residue

    imfs = np.vstack(modes)
    residue = values - imfs.sum(axis=0)
    elapsed = time.perf_counter() - start
    return DecompositionResult(imfs=imfs, residue=residue, elapsed_seconds=elapsed)


def run_length_number(values: np.ndarray) -> int:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return 0
    symbols = np.where(values >= values.mean(), 1, -1)
    return int(1 + np.sum(symbols[1:] != symbols[:-1]))


def mean_period(values: np.ndarray) -> float:
    peaks, _ = find_peaks(np.asarray(values, dtype=float))
    if len(peaks) == 0:
        return np.inf
    return float(len(values) / len(peaks))


def spearman_significance(pvalue: float) -> str:
    if pd.isna(pvalue):
        return ""
    if pvalue < 0.01:
        return "***"
    if pvalue < 0.05:
        return "**"
    if pvalue < 0.10:
        return "*"
    return ""


def safe_spearman_stats(x: np.ndarray, y: np.ndarray) -> dict[str, float | str]:
    result = spearmanr(x, y, nan_policy="omit")
    corr = float(result.correlation) if pd.notna(result.correlation) else np.nan
    pvalue = float(result.pvalue) if pd.notna(result.pvalue) else np.nan
    return {
        "spearman_corr": corr,
        "spearman_pvalue": pvalue,
        "spearman_significance": spearman_significance(pvalue),
    }


def safe_spearman(x: np.ndarray, y: np.ndarray) -> float:
    return float(safe_spearman_stats(x, y)["spearman_corr"])

In [4]:
decomposition_results: dict[str, DecompositionResult] = {}
reconstruction_errors: dict[str, float] = {}

for series_idx, series_name in enumerate(RETURN_COLUMNS):
    series_seed = PARAMS["seed"] + series_idx
    values = returns[series_name].to_numpy(dtype=float)
    result = iceemdan_decompose(
        values,
        trials=PARAMS["trials"],
        epsilon=PARAMS["epsilon"],
        seed=series_seed,
        spline_kind=PARAMS["spline_kind"],
        max_imf=PARAMS["max_imf"],
    )
    decomposition_results[series_name] = result
    recon_error = float(np.max(np.abs(result.imfs.sum(axis=0) + result.residue - values)))
    reconstruction_errors[series_name] = recon_error
    print(
        f"{series_name}: {result.imfs.shape[0]} IMFs + residue, "
        f"{result.elapsed_seconds:.2f}s, max reconstruction error={recon_error:.3e}"
    )

max_reconstruction_error = max(reconstruction_errors.values())
assert max_reconstruction_error < PARAMS["reconstruction_tolerance"], reconstruction_errors

gold_futures: 10 IMFs + residue, 7.77s, max reconstruction error=6.939e-18
silver_futures: 10 IMFs + residue, 7.84s, max reconstruction error=6.939e-18
platinum_futures: 10 IMFs + residue, 7.38s, max reconstruction error=3.469e-18
palladium_futures: 9 IMFs + residue, 7.86s, max reconstruction error=1.735e-18
gold_spot: 10 IMFs + residue, 8.24s, max reconstruction error=6.939e-18
silver_spot: 10 IMFs + residue, 7.68s, max reconstruction error=8.674e-19
platinum_spot: 10 IMFs + residue, 7.44s, max reconstruction error=3.469e-18
palladium_spot: 10 IMFs + residue, 7.78s, max reconstruction error=6.939e-18


## ICEEMDAN IMF Long Table

The decomposition output is converted to a long table for downstream diagnostics, visualization, and reproducibility.


In [5]:
imf_rows = []

for series_name, result in decomposition_results.items():
    for imf_idx, mode in enumerate(result.imfs, start=1):
        imf_rows.extend(
            {"date": date, "series": series_name, "mode": f"IMF{imf_idx}", "mode_order": imf_idx, "value": value}
            for date, value in zip(returns["date"], mode)
        )
    residue_order = result.imfs.shape[0] + 1
    imf_rows.extend(
        {"date": date, "series": series_name, "mode": "residue", "mode_order": residue_order, "value": value}
        for date, value in zip(returns["date"], result.residue)
    )

iceemdan_imfs_long = pd.DataFrame(imf_rows)
assert iceemdan_imfs_long["series"].nunique() == len(RETURN_COLUMNS)
assert iceemdan_imfs_long.loc[iceemdan_imfs_long["mode"].eq("residue"), "series"].nunique() == len(RETURN_COLUMNS)

print("ICEEMDAN IMF long shape:", iceemdan_imfs_long.shape)
display(iceemdan_imfs_long.head())


ICEEMDAN IMF long shape: (269700, 5)


,date,series,mode,mode_order,value
0,2012-01-04,gold_futures,IMF1,1,0.388860
1,2012-01-05,gold_futures,IMF1,1,0.133993
2,2012-01-06,gold_futures,IMF1,1,-0.240092
3,2012-01-09,gold_futures,IMF1,1,-0.690347
4,2012-01-10,gold_futures,IMF1,1,0.766371


## Hierarchical Ward-RLN Reconstruction

Ward hierarchical clustering is applied to standardized `[log(mean period), log(RLN/T)]` IMF features. Three clusters are mapped to STC/MTC/LTC by increasing cluster mean period, and the residue is added to LTC.


In [6]:
HIERARCHICAL_DENDROGRAM_DIR = OUTPUT_DIR / "visualizations" / "hierarchical_ward_dendrograms"
HIERARCHICAL_DENDROGRAM_DIR.mkdir(parents=True, exist_ok=True)

hierarchical_diagnostic_rows = []
hierarchical_component_long_rows = []
hierarchical_component_wide = pd.DataFrame({"date": returns["date"]})
hierarchical_ward_reconstruction_errors: dict[str, float] = {}
hierarchical_ward_scale_counts: dict[str, dict[str, int]] = {}
hierarchical_ward_dendrogram_paths: dict[str, str] = {}
hierarchical_ward_linkage_matrices: dict[str, np.ndarray] = {}

feature_epsilon = np.finfo(float).tiny
hierarchical_feature_columns = ["log_mean_period", "log_rln_ratio"]

for series_name, result in decomposition_results.items():
    original = returns[series_name].to_numpy(dtype=float)
    n_obs = original.size
    n_imfs = result.imfs.shape[0]
    if n_imfs < PARAMS["hierarchical_clusters"]:
        raise ValueError(
            f"{series_name} produced fewer than {PARAMS['hierarchical_clusters']} IMFs "
            "and cannot be reconstructed with hierarchical 3-scale groups."
        )

    component_variances = [np.var(mode, ddof=0) for mode in result.imfs]
    component_variances.append(np.var(result.residue, ddof=0))
    variance_denominator = float(np.sum(component_variances))

    imf_periods = np.array([mean_period(mode) for mode in result.imfs], dtype=float)
    imf_run_lengths = np.array([run_length_number(mode) for mode in result.imfs], dtype=float)
    imf_rln_ratios = imf_run_lengths / float(n_obs)
    feature_periods = np.where(np.isfinite(imf_periods), imf_periods, float(n_obs))
    raw_features = np.column_stack(
        [
            np.log(np.maximum(feature_periods, feature_epsilon)),
            np.log(np.maximum(imf_rln_ratios, feature_epsilon)),
        ]
    )
    scaled_features = StandardScaler().fit_transform(raw_features)

    linkage_matrix = linkage(scaled_features, method=PARAMS["hierarchical_linkage"])
    cluster_labels = fcluster(
        linkage_matrix,
        t=PARAMS["hierarchical_clusters"],
        criterion="maxclust",
    )
    observed_labels = sorted(set(int(label) for label in cluster_labels))
    if len(observed_labels) != PARAMS["hierarchical_clusters"]:
        raise ValueError(
            f"{series_name} hierarchical clustering assigned fewer than 3 non-empty scale groups: {observed_labels}"
        )

    label_mean_periods = {
        label: float(feature_periods[cluster_labels == label].mean())
        for label in observed_labels
    }
    label_mean_rln_ratios = {
        label: float(imf_rln_ratios[cluster_labels == label].mean())
        for label in observed_labels
    }
    ordered_labels = sorted(label_mean_periods, key=label_mean_periods.get)
    label_to_scale = dict(zip(ordered_labels, ["STC", "MTC", "LTC"]))

    scale_components = {
        "STC": np.zeros_like(original),
        "MTC": np.zeros_like(original),
        "LTC": np.zeros_like(original),
    }
    scale_counts = {scale: 0 for scale in ["STC", "MTC", "LTC"]}

    for imf_idx, mode in enumerate(result.imfs, start=1):
        mode_name = f"IMF{imf_idx}"
        cluster_label = int(cluster_labels[imf_idx - 1])
        scale = label_to_scale[cluster_label]
        scale_components[scale] += mode
        scale_counts[scale] += 1
        variance_contribution = float(np.var(mode, ddof=0) / variance_denominator) if variance_denominator > 0 else np.nan

        hierarchical_diagnostic_rows.append(
            {
                "series": series_name,
                "mode": mode_name,
                "mode_order": imf_idx,
                "mean_period": imf_periods[imf_idx - 1],
                **safe_spearman_stats(mode, original),
                "variance_contribution": variance_contribution,
                "run_length_number": int(imf_run_lengths[imf_idx - 1]),
                "rln_ratio": imf_rln_ratios[imf_idx - 1],
                "log_mean_period": raw_features[imf_idx - 1, 0],
                "log_rln_ratio": raw_features[imf_idx - 1, 1],
                "scaled_log_mean_period": scaled_features[imf_idx - 1, 0],
                "scaled_log_rln_ratio": scaled_features[imf_idx - 1, 1],
                "hierarchical_cluster": cluster_label,
                "hierarchical_cluster_mean_period": label_mean_periods[cluster_label],
                "hierarchical_cluster_mean_rln_ratio": label_mean_rln_ratios[cluster_label],
                "scale": scale,
            }
        )

    if any(count == 0 for count in scale_counts.values()):
        raise ValueError(f"{series_name} has an empty hierarchical Ward-RLN scale group: {scale_counts}")

    scale_components["LTC"] += result.residue
    residue_order = n_imfs + 1
    hierarchical_diagnostic_rows.append(
        {
            "series": series_name,
            "mode": "residue",
            "mode_order": residue_order,
            "mean_period": np.inf,
            **safe_spearman_stats(result.residue, original),
            "variance_contribution": float(np.var(result.residue, ddof=0) / variance_denominator) if variance_denominator > 0 else np.nan,
            "run_length_number": np.nan,
            "rln_ratio": np.nan,
            "log_mean_period": np.nan,
            "log_rln_ratio": np.nan,
            "scaled_log_mean_period": np.nan,
            "scaled_log_rln_ratio": np.nan,
            "hierarchical_cluster": np.nan,
            "hierarchical_cluster_mean_period": np.nan,
            "hierarchical_cluster_mean_rln_ratio": np.nan,
            "scale": "LTC",
        }
    )

    reconstructed = scale_components["STC"] + scale_components["MTC"] + scale_components["LTC"]
    scale_error = float(np.max(np.abs(reconstructed - original)))
    hierarchical_ward_reconstruction_errors[series_name] = scale_error
    assert scale_error < PARAMS["reconstruction_tolerance"], (series_name, scale_error)
    hierarchical_ward_scale_counts[series_name] = scale_counts

    for scale in ["STC", "MTC", "LTC"]:
        values = scale_components[scale]
        hierarchical_component_wide[f"{series_name}_{scale}"] = values
        hierarchical_component_long_rows.extend(
            {"date": date, "series": series_name, "scale": scale, "value": value}
            for date, value in zip(returns["date"], values)
        )

    fig, ax = plt.subplots(figsize=(7.2, 4.2), constrained_layout=True)
    dendrogram(
        linkage_matrix,
        labels=[f"IMF{i}" for i in range(1, n_imfs + 1)],
        ax=ax,
        color_threshold=None,
    )
    ax.set_title(f"{series_name}: Ward dendrogram")
    ax.set_xlabel("IMF")
    ax.set_ylabel("Ward distance")
    ax.grid(axis="x", visible=False)
    dendrogram_path = HIERARCHICAL_DENDROGRAM_DIR / f"{series_name}_hierarchical_ward_dendrogram.png"
    fig.savefig(dendrogram_path, bbox_inches="tight")
    dendrogram_pdf_path = dendrogram_path.with_suffix(".pdf")
    fig.savefig(dendrogram_pdf_path, bbox_inches="tight")
    plt.close(fig)
    hierarchical_ward_dendrogram_paths[series_name] = {
        "png": str(dendrogram_path.relative_to(PROJECT_ROOT)),
        "pdf": str(dendrogram_pdf_path.relative_to(PROJECT_ROOT)),
    }
    hierarchical_ward_linkage_matrices[series_name] = linkage_matrix

hierarchical_ward_diagnostics = pd.DataFrame(hierarchical_diagnostic_rows)
hierarchical_ward_components_long = pd.DataFrame(hierarchical_component_long_rows)
hierarchical_ward_components_wide = hierarchical_component_wide.copy()

expected_wide_shape = (len(returns), 1 + len(RETURN_COLUMNS) * 3)
assert hierarchical_ward_components_wide.shape == expected_wide_shape, hierarchical_ward_components_wide.shape
assert hierarchical_ward_components_wide.drop(columns=["date"]).shape[1] == len(RETURN_COLUMNS) * 3
assert max(hierarchical_ward_reconstruction_errors.values()) < PARAMS["reconstruction_tolerance"]

combined_dendrogram_pdf = HIERARCHICAL_DENDROGRAM_DIR / "hierarchical_ward_dendrograms.pdf"
with PdfPages(combined_dendrogram_pdf) as pdf:
    for series_name in RETURN_COLUMNS:
        image = plt.imread(HIERARCHICAL_DENDROGRAM_DIR / f"{series_name}_hierarchical_ward_dendrogram.png")
        fig, ax = plt.subplots(figsize=(8.0, 5.0))
        ax.imshow(image)
        ax.axis("off")
        fig.tight_layout(pad=0)
        pdf.savefig(fig, bbox_inches="tight", pad_inches=0)
        plt.close(fig)

print("Hierarchical Ward-RLN scale counts by series:")
display(pd.DataFrame(hierarchical_ward_scale_counts).T)
print("Max hierarchical Ward-RLN reconstruction error:", max(hierarchical_ward_reconstruction_errors.values()))
print("Dendrogram directory:", HIERARCHICAL_DENDROGRAM_DIR.relative_to(PROJECT_ROOT))


Hierarchical Ward-RLN scale counts by series:


,STC,MTC,LTC
gold_futures,2,4,4
silver_futures,5,3,2
platinum_futures,3,3,4
palladium_futures,3,4,2
gold_spot,3,4,3
silver_spot,3,3,4
platinum_spot,3,3,4
palladium_spot,3,3,4


Max hierarchical Ward-RLN reconstruction error: 7.105427357601002e-15
Dendrogram directory: data\processed\02_iceemdan_hierarchical_ward\visualizations\hierarchical_ward_dendrograms


## Descriptive Statistics of Reconstructed Components

Descriptive statistics are generated for the hierarchical Ward reconstructed STC, MTC, and LTC components.


In [7]:
DESCRIPTIVE_SERIES_ORDER = [
    "gold_futures",
    "gold_spot",
    "silver_futures",
    "silver_spot",
    "platinum_futures",
    "platinum_spot",
    "palladium_futures",
    "palladium_spot",
]

METAL_SERIES_PAIRS = [
    ("\u9ec4\u91d1", "gold_futures", "gold_spot"),
    ("\u767d\u94f6", "silver_futures", "silver_spot"),
    ("\u94c2\u91d1", "platinum_futures", "platinum_spot"),
    ("\u94af\u91d1", "palladium_futures", "palladium_spot"),
]

SCALE_DISPLAY_NAMES = {
    "STC": "\u77ed\u671f",
    "MTC": "\u4e2d\u671f",
    "LTC": "\u957f\u671f",
}
TABLE_HEADER_FIRST_COLUMN = "\u7edf\u8ba1\u91cf"
FUTURES_LABEL = "\u671f\u8d27"
SPOT_LABEL = "\u73b0\u8d27"
TABLE_CAPTION_TEMPLATE = "\u8d35\u91d1\u5c5e\u671f\u8d27\u4e0e\u73b0\u8d27\u6536\u76ca\u7387{scale_name}\u91cd\u6784\u6210\u5206\u63cf\u8ff0\u7edf\u8ba1"
TABLE_NOTE_TEMPLATE = (
    "\u6ce8\uff1a\u8be5\u8868\u62a5\u544a{scale_name}\u91cd\u6784\u6210\u5206\uff08{scale}\uff09\u7684\u63cf\u8ff0\u7edf\u8ba1\uff1b"
    "Jarque-Bera\u68c0\u9a8c\u7528\u4e8e\u68c0\u9a8c\u6b63\u6001\u6027\uff0cADF\u68c0\u9a8c\u7528\u4e8e\u68c0\u9a8c\u5e73\u7a33\u6027\uff1b"
    "$^{{*}}$\u3001$^{{**}}$\u548c$^{{***}}$\u5206\u522b\u8868\u793a\u572810\\%\u30015\\%\u548c1\\%\u6c34\u5e73\u4e0a\u663e\u8457\u3002"
)

RECONSTRUCTED_COMPONENT_STATS_PATH = OUTPUT_DIR / "hierarchical_ward_reconstructed_component_descriptive_stats.csv"
RECONSTRUCTED_COMPONENT_STATS_LATEX_PATH = OUTPUT_DIR / "hierarchical_ward_reconstructed_component_descriptive_stats.tex"


def safe_adf(series: pd.Series) -> pd.Series:
    clean = series.dropna()
    try:
        result = adfuller(clean, autolag="AIC")
        return pd.Series(
            {
                "adf_stat": result[0],
                "adf_pvalue": result[1],
                "adf_lags": result[2],
                "adf_nobs": result[3],
            }
        )
    except Exception:
        return pd.Series({"adf_stat": np.nan, "adf_pvalue": np.nan, "adf_lags": np.nan, "adf_nobs": np.nan})


def describe_component_series(series: pd.Series) -> pd.Series:
    clean = series.dropna()
    jb_stat, jb_pvalue = stats.jarque_bera(clean)
    adf_values = safe_adf(clean)
    return pd.Series(
        {
            "n_obs": clean.shape[0],
            "max": clean.max(),
            "min": clean.min(),
            "mean": clean.mean(),
            "std": clean.std(ddof=1),
            "skew": stats.skew(clean, bias=False),
            "kurtosis": stats.kurtosis(clean, fisher=False, bias=False),
            "jarque_bera": jb_stat,
            "jb_pvalue": jb_pvalue,
            **adf_values.to_dict(),
        }
    )


def significance_stars(pvalue: float) -> str:
    if pd.isna(pvalue):
        return ""
    if pvalue < 0.01:
        return "$^{***}$"
    if pvalue < 0.05:
        return "$^{**}$"
    if pvalue < 0.10:
        return "$^{*}$"
    return ""


def format_value(value: float, digits: int = 3) -> str:
    if pd.isna(value):
        return ""
    rounded = f"{value:.{digits}f}"
    return "0.000" if rounded == "-0.000" and digits == 3 else rounded


def format_test_stat(value: float, pvalue: float, digits: int = 2) -> str:
    if pd.isna(value):
        return ""
    return f"{value:.{digits}f}{significance_stars(pvalue)}"


component_stats_rows = []
for scale in ["STC", "MTC", "LTC"]:
    for series_name in DESCRIPTIVE_SERIES_ORDER:
        values = hierarchical_ward_components_long.loc[
            hierarchical_ward_components_long["scale"].eq(scale) & hierarchical_ward_components_long["series"].eq(series_name),
            "value",
        ]
        stats_row = describe_component_series(values).to_dict()
        component_stats_rows.append({"scale": scale, "series": series_name, **stats_row})

hierarchical_ward_reconstructed_component_descriptive_stats = pd.DataFrame(component_stats_rows)
assert hierarchical_ward_reconstructed_component_descriptive_stats.shape[0] == 24
assert hierarchical_ward_reconstructed_component_descriptive_stats["n_obs"].eq(len(returns)).all()


def build_reconstructed_component_latex_table(scale: str) -> str:
    scale_name = SCALE_DISPLAY_NAMES[scale]
    caption = TABLE_CAPTION_TEMPLATE.format(scale_name=scale_name)
    note = TABLE_NOTE_TEMPLATE.format(scale_name=scale_name, scale=scale)
    panel = (
        hierarchical_ward_reconstructed_component_descriptive_stats
        .loc[hierarchical_ward_reconstructed_component_descriptive_stats["scale"].eq(scale)]
        .set_index("series")
    )
    statistic_rows = [
        ("Max", "max", "value", 3),
        ("Min", "min", "value", 3),
        ("Mean", "mean", "value", 3),
        ("Std. Dev.", "std", "value", 3),
        ("Skew.", "skew", "value", 3),
        ("Kurt.", "kurtosis", "value", 3),
        ("Jarque-Bera", "jarque_bera", "jb", 1),
        ("ADF", "adf_stat", "adf", 2),
    ]
    latex_lines = [
        r"\begin{table}[H]",
        rf"\caption{{{caption}}}",
        rf"\label{{tab:hierarchical-reconstructed-descriptive-{scale.lower()}}}",
        r"\begin{normaltable}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{lrrrrrrrr}",
        r"\toprule",
        " & " + " & ".join(rf"\multicolumn{{2}}{{c}}{{{metal}}}" for metal, _, _ in METAL_SERIES_PAIRS) + r" \\",
        r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}\cmidrule(lr){8-9}",
        f"{TABLE_HEADER_FIRST_COLUMN} & {FUTURES_LABEL} & {SPOT_LABEL} & {FUTURES_LABEL} & {SPOT_LABEL} & {FUTURES_LABEL} & {SPOT_LABEL} & {FUTURES_LABEL} & {SPOT_LABEL} " + r"\\",
        r"\midrule",
    ]
    for label, column, kind, digits in statistic_rows:
        values = []
        for _, futures_series, spot_series in METAL_SERIES_PAIRS:
            for series_name in [futures_series, spot_series]:
                if kind == "jb":
                    values.append(format_test_stat(panel.loc[series_name, column], panel.loc[series_name, "jb_pvalue"], digits=digits))
                elif kind == "adf":
                    values.append(format_test_stat(panel.loc[series_name, column], panel.loc[series_name, "adf_pvalue"], digits=digits))
                else:
                    values.append(format_value(panel.loc[series_name, column], digits=digits))
        latex_lines.append(label + " & " + " & ".join(values) + r" \\")
    latex_lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}}",
            r"\end{normaltable}",
            r"\begin{flushleft}",
            rf"{{\songti\zihao{{5}}{note}}}",
            r"\end{flushleft}",
            r"\end{table}",
        ]
    )
    return "\n".join(latex_lines)


hierarchical_ward_reconstructed_component_descriptive_stats_latex = "\n\n".join(
    build_reconstructed_component_latex_table(scale) for scale in ["STC", "MTC", "LTC"]
) + "\n"

print("All reconstructed component JB tests significant at 1%:", hierarchical_ward_reconstructed_component_descriptive_stats["jb_pvalue"].lt(0.01).all())
print("All reconstructed component ADF tests significant at 1%:", hierarchical_ward_reconstructed_component_descriptive_stats["adf_pvalue"].lt(0.01).all())
display(hierarchical_ward_reconstructed_component_descriptive_stats)


All reconstructed component JB tests significant at 1%: True
All reconstructed component ADF tests significant at 1%: False


,scale,series,n_obs,max,min,mean,std,skew,kurtosis,jarque_bera,jb_pvalue,adf_stat,adf_pvalue,adf_lags,adf_nobs
0,STC,gold_futures,3100.0,6.910271,-8.766475,0.008191,1.013101,-0.124577,8.020202,3250.331858,0.000000e+00,-16.419122,2.540376e-29,15.0,3084.0
1,STC,gold_spot,3100.0,6.631884,-9.820696,0.007955,1.023987,-0.403977,9.234286,5085.185477,0.000000e+00,-26.674353,0.000000e+00,9.0,3090.0
2,STC,silver_futures,3100.0,16.935875,-37.410514,0.022428,2.204495,-1.715545,36.575155,146641.686852,0.000000e+00,-16.728898,1.388451e-29,19.0,3080.0
3,STC,silver_spot,3100.0,15.613034,-24.979810,0.038369,1.989150,-0.496941,19.931244,37027.547977,0.000000e+00,-12.106548,1.962956e-22,20.0,3079.0
4,STC,platinum_futures,3100.0,11.315514,-15.856042,0.014596,1.757856,-0.357915,9.324133,5212.299864,0.000000e+00,-28.320604,0.000000e+00,8.0,3091.0
5,STC,platinum_spot,3100.0,9.516410,-15.208384,0.017094,1.642707,-0.417687,9.493838,5516.212907,0.000000e+00,-13.416088,4.279420e-25,16.0,3083.0
6,STC,palladium_futures,3100.0,16.824312,-21.108451,0.009227,2.330145,-0.523934,10.969110,8314.238878,0.000000e+00,-24.996107,0.000000e+00,9.0,3090.0
7,STC,palladium_spot,3100.0,15.216458,-16.975716,0.038979,2.179869,-0.460245,9.136481,4954.555897,0.000000e+00,-25.487716,0.000000e+00,9.0,3090.0
8,MTC,gold_futures,3100.0,3.533731,-3.688464,0.003025,0.534769,-0.475034,8.636121,4203.535881,0.000000e+00,-11.737047,1.300235e-21,24.0,3075.0
9,MTC,gold_spot,3100.0,1.471408,-1.686289,-0.000565,0.367703,0.061261,3.858702,96.445784,1.140418e-21,-10.100138,1.064845e-17,23.0,3076.0


## Save Core Outputs

Core ICEEMDAN, hierarchical reconstruction, descriptive statistics, and manifest files are written to the new output namespace.


In [8]:
iceemdan_imfs_long.to_csv(OUTPUT_DIR / "iceemdan_imfs_long.csv", index=False, encoding="utf-8-sig")
hierarchical_ward_diagnostics.to_csv(OUTPUT_DIR / "hierarchical_ward_diagnostics.csv", index=False, encoding="utf-8-sig")
hierarchical_ward_components_long.to_csv(OUTPUT_DIR / "hierarchical_ward_components_long.csv", index=False, encoding="utf-8-sig")
hierarchical_ward_components_wide.to_csv(OUTPUT_DIR / "hierarchical_ward_components_wide.csv", index=False, encoding="utf-8-sig")
hierarchical_ward_reconstructed_component_descriptive_stats.to_csv(RECONSTRUCTED_COMPONENT_STATS_PATH, index=False, encoding="utf-8-sig")
RECONSTRUCTED_COMPONENT_STATS_LATEX_PATH.write_text(hierarchical_ward_reconstructed_component_descriptive_stats_latex, encoding="utf-8")

manifest = {
    "method": "Direct ICEEMDAN implementation using PyEMD.EMD local-mean operator, followed by Ward hierarchical clustering reconstruction.",
    "hierarchical_ward_note": "Ward linkage on standardized [log(mean_period), log(run_length_number / T)] IMF features; clusters are cut into three groups and mapped to STC/MTC/LTC by increasing cluster mean period. The residue is added to LTC.",
    "parameters": PARAMS,
    "input_checks": input_checks,
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
    "outputs": {
        "iceemdan_imfs_long": "data/processed/02_iceemdan_hierarchical_ward/iceemdan_imfs_long.csv",
        "hierarchical_ward_diagnostics": "data/processed/02_iceemdan_hierarchical_ward/hierarchical_ward_diagnostics.csv",
        "hierarchical_ward_components_long": "data/processed/02_iceemdan_hierarchical_ward/hierarchical_ward_components_long.csv",
        "hierarchical_ward_components_wide": "data/processed/02_iceemdan_hierarchical_ward/hierarchical_ward_components_wide.csv",
        "hierarchical_ward_reconstructed_component_descriptive_stats": "data/processed/02_iceemdan_hierarchical_ward/hierarchical_ward_reconstructed_component_descriptive_stats.csv",
        "hierarchical_ward_reconstructed_component_descriptive_stats_latex": "data/processed/02_iceemdan_hierarchical_ward/hierarchical_ward_reconstructed_component_descriptive_stats.tex",
        "hierarchical_ward_reconstruction_error_summary": "data/processed/02_iceemdan_hierarchical_ward/hierarchical_ward_reconstruction_error_summary.csv",
        "hierarchical_ward_dendrograms": "data/processed/02_iceemdan_hierarchical_ward/visualizations/hierarchical_ward_dendrograms",
        "imf_time_curves": "data/processed/02_iceemdan_hierarchical_ward/visualizations/imf_time_curves",
        "hierarchical_ward_reconstruction_curves": "data/processed/02_iceemdan_hierarchical_ward/visualizations/hierarchical_ward_reconstruction_curves",
        "hierarchical_ward_reconstruction_data": "data/processed/02_iceemdan_hierarchical_ward/visualizations/hierarchical_ward_reconstruction_data",
    },
    "spearman_significance_note": "spearman_pvalue is the two-sided p-value from scipy.stats.spearmanr; *, **, and *** indicate p < 0.10, p < 0.05, and p < 0.01.",
    "n_imfs_by_series": {series: int(result.imfs.shape[0]) for series, result in decomposition_results.items()},
    "decomposition_seconds_by_series": {series: round(float(result.elapsed_seconds), 4) for series, result in decomposition_results.items()},
    "iceemdan_reconstruction_error_by_series": reconstruction_errors,
    "hierarchical_ward_reconstruction_error_by_series": hierarchical_ward_reconstruction_errors,
    "hierarchical_ward_scale_counts_by_series": hierarchical_ward_scale_counts,
    "hierarchical_ward_dendrogram_paths": hierarchical_ward_dendrogram_paths,
    "limitation_note": input_checks["limitation_note"],
}

with open(OUTPUT_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

for path in [
    OUTPUT_DIR / "iceemdan_imfs_long.csv",
    OUTPUT_DIR / "hierarchical_ward_diagnostics.csv",
    OUTPUT_DIR / "hierarchical_ward_components_long.csv",
    OUTPUT_DIR / "hierarchical_ward_components_wide.csv",
    OUTPUT_DIR / "hierarchical_ward_reconstructed_component_descriptive_stats.csv",
    OUTPUT_DIR / "hierarchical_ward_reconstructed_component_descriptive_stats.tex",
    OUTPUT_DIR / "run_manifest.json",
]:
    print(path.relative_to(PROJECT_ROOT), path.stat().st_size)
print(HIERARCHICAL_DENDROGRAM_DIR.relative_to(PROJECT_ROOT), len(list(HIERARCHICAL_DENDROGRAM_DIR.glob("*_hierarchical_ward_dendrogram.png"))))


data\processed\02_iceemdan_hierarchical_ward\iceemdan_imfs_long.csv 14626464
data\processed\02_iceemdan_hierarchical_ward\hierarchical_ward_diagnostics.csv 21002
data\processed\02_iceemdan_hierarchical_ward\hierarchical_ward_components_long.csv 3738823
data\processed\02_iceemdan_hierarchical_ward\hierarchical_ward_components_wide.csv 1525842
data\processed\02_iceemdan_hierarchical_ward\hierarchical_ward_reconstructed_component_descriptive_stats.csv 5492
data\processed\02_iceemdan_hierarchical_ward\hierarchical_ward_reconstructed_component_descriptive_stats.tex 4823
data\processed\02_iceemdan_hierarchical_ward\run_manifest.json 8438
data\processed\02_iceemdan_hierarchical_ward\visualizations\hierarchical_ward_dendrograms 8


## Hierarchical Summary

The table below summarizes the IMF groups assigned to each hierarchical Ward scale.


In [9]:
display(hierarchical_ward_diagnostics.head(20))

hierarchical_summary = (
    hierarchical_ward_diagnostics.loc[hierarchical_ward_diagnostics["mode"].str.startswith("IMF")]
    .groupby(["series", "scale"])
    .agg(
        n_modes=("mode", "count"),
        min_mode_order=("mode_order", "min"),
        max_mode_order=("mode_order", "max"),
        mean_rln=("run_length_number", "mean"),
        mean_period=("mean_period", "mean"),
        variance_contribution=("variance_contribution", "sum"),
    )
    .reset_index()
    .sort_values(["series", "scale"])
)
display(hierarchical_summary)

,series,mode,mode_order,mean_period,spearman_corr,spearman_pvalue,spearman_significance,variance_contribution,run_length_number,rln_ratio,log_mean_period,log_rln_ratio,scaled_log_mean_period,scaled_log_rln_ratio,hierarchical_cluster,hierarchical_cluster_mean_period,hierarchical_cluster_mean_rln_ratio,scale
0,gold_futures,IMF1,1,2.716915,0.738333,0.000000e+00,***,0.580444,2181.0,0.703548,0.999497,-0.351619,-1.500634,1.561104,2.0,4.101820,0.526129,STC
1,gold_futures,IMF2,2,5.486726,0.425427,1.634599e-136,***,0.208505,1081.0,0.348710,1.702332,-1.053516,-1.176100,1.219609,2.0,4.101820,0.526129,STC
2,gold_futures,IMF3,3,11.032028,0.273024,4.025686e-54,***,0.109131,519.0,0.167419,2.400803,-1.787254,-0.853582,0.862621,3.0,40.616142,0.079435,MTC
3,gold_futures,IMF4,4,21.527778,0.219481,3.976728e-35,***,0.057473,266.0,0.085806,3.069344,-2.455661,-0.544883,0.537419,3.0,40.616142,0.079435,MTC
4,gold_futures,IMF5,5,41.333333,0.160755,2.147797e-19,***,0.023842,134.0,0.043226,3.721669,-3.141318,-0.243672,0.203825,3.0,40.616142,0.079435,MTC
5,gold_futures,IMF6,6,88.571429,0.090253,4.820728e-07,***,0.013120,66.0,0.021290,4.483809,-3.849503,0.108246,-0.140730,3.0,40.616142,0.079435,MTC
6,gold_futures,IMF7,7,206.666667,0.066355,2.181173e-04,***,0.004217,27.0,0.008710,5.331107,-4.743321,0.499486,-0.575602,1.0,1131.130952,0.004113,LTC
7,gold_futures,IMF8,8,442.857143,0.033870,5.934950e-02,*,0.000946,13.0,0.004194,6.093247,-5.474208,0.851404,-0.931203,1.0,1131.130952,0.004113,LTC
8,gold_futures,IMF9,9,775.000000,0.054364,2.462874e-03,***,0.001395,7.0,0.002258,6.652863,-6.093247,1.109807,-1.232386,1.0,1131.130952,0.004113,LTC
9,gold_futures,IMF10,10,3100.000000,0.018743,2.968434e-01,,0.000251,4.0,0.001290,8.039157,-6.652863,1.749928,-1.504657,1.0,1131.130952,0.004113,LTC


,series,scale,n_modes,min_mode_order,max_mode_order,mean_rln,mean_period,variance_contribution
0,gold_futures,LTC,4,7,10,12.750000,1131.130952,0.006808
1,gold_futures,MTC,4,3,6,246.250000,40.616142,0.203566
2,gold_futures,STC,2,1,2,1631.000000,4.101820,0.788949
3,gold_spot,LTC,3,8,10,8.333333,922.619048,0.004577
4,gold_spot,MTC,4,4,7,124.750000,90.102337,0.111163
5,gold_spot,STC,3,1,3,1264.666667,6.258501,0.883848
6,palladium_futures,LTC,2,8,9,6.500000,1085.000000,0.003195
7,palladium_futures,MTC,4,4,7,117.250000,103.426774,0.083329
8,palladium_futures,STC,3,1,3,1242.666667,6.258171,0.913436
9,palladium_spot,LTC,4,7,10,11.000000,1337.797619,0.005919


## Visualization Exports

This section exports IMF time curves, hierarchical reconstruction data, individual reconstruction figures, and a 4x2 reconstruction panel.


In [10]:
IMF_TIME_CURVE_DIR = OUTPUT_DIR / "visualizations" / "imf_time_curves"
HIERARCHICAL_RECONSTRUCTION_CURVE_DIR = OUTPUT_DIR / "visualizations" / "hierarchical_ward_reconstruction_curves"
HIERARCHICAL_RECONSTRUCTION_DATA_DIR = OUTPUT_DIR / "visualizations" / "hierarchical_ward_reconstruction_data"
HIERARCHICAL_RECONSTRUCTION_CURVE_DIR.mkdir(parents=True, exist_ok=True)
IMF_TIME_CURVE_DIR.mkdir(parents=True, exist_ok=True)
HIERARCHICAL_RECONSTRUCTION_DATA_DIR.mkdir(parents=True, exist_ok=True)


def ensure_hierarchical_reconstruction_inputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if "hierarchical_ward_components_wide" in globals():
        hierarchical_components = hierarchical_ward_components_wide.copy()
    else:
        hierarchical_components = pd.read_csv(
            OUTPUT_DIR / "hierarchical_ward_components_wide.csv",
            parse_dates=["date"],
        )

    if "hierarchical_ward_diagnostics" in globals():
        hierarchical_diagnostics = hierarchical_ward_diagnostics.copy()
    else:
        hierarchical_diagnostics = pd.read_csv(OUTPUT_DIR / "hierarchical_ward_diagnostics.csv")

    if "returns" in globals():
        original_returns = returns.copy()
    else:
        original_returns = pd.read_csv(DATA_PATH, parse_dates=["date"])

    for frame in [hierarchical_components, original_returns]:
        frame["date"] = pd.to_datetime(frame["date"])

    return hierarchical_components, hierarchical_diagnostics, original_returns


hierarchical_components_for_plot, hierarchical_diagnostics_for_plot, hierarchical_returns_for_plot = ensure_hierarchical_reconstruction_inputs()


def build_hierarchical_reconstruction(series_name: str) -> pd.DataFrame:
    scale_columns = [f"{series_name}_{scale}" for scale in ["STC", "MTC", "LTC"]]
    missing_columns = [column for column in scale_columns if column not in hierarchical_components_for_plot.columns]
    if missing_columns:
        raise ValueError(f"Missing hierarchical Ward-RLN component columns for {series_name}: {missing_columns}")

    original = hierarchical_returns_for_plot[["date", series_name]].rename(columns={series_name: "original"})
    scales = hierarchical_components_for_plot[["date", *scale_columns]].copy()
    rename_map = {f"{series_name}_{scale}": scale for scale in ["STC", "MTC", "LTC"]}
    result = original.merge(scales.rename(columns=rename_map), on="date", how="inner")
    result["hierarchical_reconstructed"] = result[["STC", "MTC", "LTC"]].sum(axis=1)
    result["hierarchical_error"] = result["hierarchical_reconstructed"] - result["original"]
    return result


def plot_hierarchical_reconstruction(series_name: str, save: bool = True, show: bool = False) -> pd.DataFrame:
    reconstructed = build_hierarchical_reconstruction(series_name)

    fig, axes = plt.subplots(3, 1, figsize=(12.5, 7.8), sharex=True, constrained_layout=True)
    axes = np.atleast_1d(axes)

    axes[0].plot(
        reconstructed["date"],
        reconstructed["original"],
        label="Original",
        color=SERIES_COLORS["original"],
        linewidth=0.8,
    )
    axes[0].plot(
        reconstructed["date"],
        reconstructed["hierarchical_reconstructed"],
        label="STC + MTC + LTC",
        color=SERIES_COLORS["scale_reconstructed"],
        linewidth=0.8,
        alpha=0.85,
    )
    axes[0].set_title(f"{series_name}: original vs hierarchical Ward-RLN reconstructed")
    axes[0].legend(loc="upper right")
    format_time_axis(axes[0], ylabel="Return (%)", zero_line=True)

    for scale in ["STC", "MTC", "LTC"]:
        axes[1].plot(
            reconstructed["date"],
            reconstructed[scale],
            label=scale,
            color=SCALE_COLORS[scale],
            linewidth=0.8,
            alpha=0.9,
        )
    axes[1].set_title("Hierarchical Ward-RLN reconstructed components")
    axes[1].legend(loc="upper right", ncols=3)
    format_time_axis(axes[1], ylabel="Return (%)", zero_line=True)

    max_error = reconstructed["hierarchical_error"].abs().max()
    axes[2].plot(
        reconstructed["date"],
        reconstructed["hierarchical_error"],
        color=SERIES_COLORS["emd_error"],
        linewidth=0.75,
    )
    axes[2].set_title(f"Reconstruction error, max abs = {max_error:.3e}")
    format_time_axis(axes[2], ylabel="Error", zero_line=True)
    axes[2].set_xlabel("Date")

    if save:
        fig.savefig(HIERARCHICAL_RECONSTRUCTION_CURVE_DIR / f"{series_name}_hierarchical_ward_reconstruction.png", bbox_inches="tight")
        fig.savefig(HIERARCHICAL_RECONSTRUCTION_CURVE_DIR / f"{series_name}_hierarchical_ward_reconstruction.pdf", bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)

    return reconstructed


def plot_hierarchical_reconstruction_panel(series_names: list[str], save: bool = True, show: bool = False) -> None:
    fig, axes = plt.subplots(4, 2, figsize=(12.5, 10.2), sharex=True, constrained_layout=True)
    axes = axes.ravel()

    for ax, series_name in zip(axes, series_names):
        reconstructed = build_hierarchical_reconstruction(series_name)
        ax.plot(
            reconstructed["date"],
            reconstructed["original"],
            label="Original",
            color=SERIES_COLORS["original"],
            linewidth=0.65,
            alpha=0.9,
        )
        ax.plot(
            reconstructed["date"],
            reconstructed["hierarchical_reconstructed"],
            label="Reconstructed",
            color=SERIES_COLORS["scale_reconstructed"],
            linewidth=0.65,
            alpha=0.85,
        )
        ax.set_title(series_name)
        format_time_axis(ax, zero_line=True)

    axes[0].legend(loc="upper right")
    fig.suptitle("Hierarchical Ward-RLN Original vs Reconstructed Curves", fontsize=12.5, fontweight="semibold")
    if save:
        fig.savefig(HIERARCHICAL_RECONSTRUCTION_CURVE_DIR / "hierarchical_ward_original_vs_reconstructed_panel.png", bbox_inches="tight")
        fig.savefig(HIERARCHICAL_RECONSTRUCTION_CURVE_DIR / "hierarchical_ward_original_vs_reconstructed_panel.pdf", bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)


imfs_for_plot = iceemdan_imfs_long.copy()
diagnostics_for_plot = hierarchical_ward_diagnostics.copy()
MODE_SCALE_LOOKUP = (
    diagnostics_for_plot[["series", "mode", "scale"]]
    .set_index(["series", "mode"])["scale"]
    .to_dict()
)


def export_imf_time_curves(series_name: str) -> None:
    series_imfs = (
        imfs_for_plot.loc[imfs_for_plot["series"].eq(series_name)]
        .sort_values(["mode_order", "date"])
        .copy()
    )
    mode_table = series_imfs[["mode", "mode_order"]].drop_duplicates().sort_values("mode_order")
    modes = mode_table["mode"].tolist()
    fig, axes = plt.subplots(
        len(modes),
        1,
        figsize=(12.5, max(3.2, 1.18 * len(modes))),
        sharex=True,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)
    for ax, mode_name in zip(axes, modes):
        mode_values = series_imfs.loc[series_imfs["mode"].eq(mode_name)]
        scale = MODE_SCALE_LOOKUP.get((series_name, mode_name), "residue" if mode_name == "residue" else "")
        color = SCALE_COLORS.get(scale, "#444444")
        ax.plot(mode_values["date"], mode_values["value"], color=color, linewidth=0.7, alpha=0.95)
        format_time_axis(ax, zero_line=True)
        ax.set_ylabel(mode_name, rotation=0, ha="right", va="center", labelpad=26)
        if scale:
            ax.text(0.01, 0.78, scale, transform=ax.transAxes, color=color, fontsize=8.8, fontweight="semibold")
    axes[0].set_title(f"{series_name}: ICEEMDAN IMF time curves")
    axes[-1].set_xlabel("Date")
    fig.savefig(IMF_TIME_CURVE_DIR / f"{series_name}_iceemdan_imf_time_curves.png", bbox_inches="tight")
    fig.savefig(IMF_TIME_CURVE_DIR / f"{series_name}_iceemdan_imf_time_curves.pdf", bbox_inches="tight")
    plt.close(fig)


In [11]:
hierarchical_reconstruction_error_summary_rows = []
hierarchical_reconstruction_data_paths: dict[str, str] = {}
hierarchical_reconstruction_figure_paths: dict[str, dict[str, str]] = {}
imf_time_curve_paths: dict[str, dict[str, str]] = {}

for series_name in RETURN_COLUMNS:
    export_imf_time_curves(series_name)
    imf_time_curve_paths[series_name] = {
        "png": str((IMF_TIME_CURVE_DIR / f"{series_name}_iceemdan_imf_time_curves.png").relative_to(PROJECT_ROOT)),
        "pdf": str((IMF_TIME_CURVE_DIR / f"{series_name}_iceemdan_imf_time_curves.pdf").relative_to(PROJECT_ROOT)),
    }
    reconstructed = plot_hierarchical_reconstruction(series_name, save=True, show=False)
    data_path = HIERARCHICAL_RECONSTRUCTION_DATA_DIR / f"{series_name}_hierarchical_ward_reconstruction.csv"
    reconstructed.to_csv(data_path, index=False, encoding="utf-8-sig")
    hierarchical_reconstruction_data_paths[series_name] = str(data_path.relative_to(PROJECT_ROOT))
    hierarchical_reconstruction_figure_paths[series_name] = {
        "png": str((HIERARCHICAL_RECONSTRUCTION_CURVE_DIR / f"{series_name}_hierarchical_ward_reconstruction.png").relative_to(PROJECT_ROOT)),
        "pdf": str((HIERARCHICAL_RECONSTRUCTION_CURVE_DIR / f"{series_name}_hierarchical_ward_reconstruction.pdf").relative_to(PROJECT_ROOT)),
    }
    hierarchical_reconstruction_error_summary_rows.append(
        {
            "series": series_name,
            "n_obs": int(len(reconstructed)),
            "max_abs_error": float(reconstructed["hierarchical_error"].abs().max()),
            "mean_abs_error": float(reconstructed["hierarchical_error"].abs().mean()),
            "rmse": float(np.sqrt(np.mean(np.square(reconstructed["hierarchical_error"])))),
        }
    )

plot_hierarchical_reconstruction_panel(RETURN_COLUMNS, save=True, show=False)
hierarchical_reconstruction_panel_paths = {
    "png": str((HIERARCHICAL_RECONSTRUCTION_CURVE_DIR / "hierarchical_ward_original_vs_reconstructed_panel.png").relative_to(PROJECT_ROOT)),
    "pdf": str((HIERARCHICAL_RECONSTRUCTION_CURVE_DIR / "hierarchical_ward_original_vs_reconstructed_panel.pdf").relative_to(PROJECT_ROOT)),
}

hierarchical_ward_reconstruction_error_summary = pd.DataFrame(hierarchical_reconstruction_error_summary_rows)
hierarchical_ward_reconstruction_error_summary.to_csv(
    OUTPUT_DIR / "hierarchical_ward_reconstruction_error_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Hierarchical Ward-RLN reconstruction data directory:", HIERARCHICAL_RECONSTRUCTION_DATA_DIR.relative_to(PROJECT_ROOT))
print("Hierarchical Ward-RLN reconstruction curve directory:", HIERARCHICAL_RECONSTRUCTION_CURVE_DIR.relative_to(PROJECT_ROOT))
display(hierarchical_ward_reconstruction_error_summary)


# Persist the hierarchical reconstruction visualization outputs back to the run manifest.
manifest_path = OUTPUT_DIR / "run_manifest.json"
if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
else:
    manifest = {"outputs": {}}
manifest.setdefault("outputs", {}).update(
    {
        "imf_time_curves": "data/processed/02_iceemdan_hierarchical_ward/visualizations/imf_time_curves",
        "hierarchical_ward_reconstruction_curves": "data/processed/02_iceemdan_hierarchical_ward/visualizations/hierarchical_ward_reconstruction_curves",
        "hierarchical_ward_reconstruction_data": "data/processed/02_iceemdan_hierarchical_ward/visualizations/hierarchical_ward_reconstruction_data",
        "hierarchical_ward_reconstruction_error_summary": "data/processed/02_iceemdan_hierarchical_ward/hierarchical_ward_reconstruction_error_summary.csv",
    }
)
manifest["imf_time_curve_paths"] = imf_time_curve_paths
manifest["hierarchical_ward_reconstruction_data_paths"] = hierarchical_reconstruction_data_paths
manifest["hierarchical_ward_reconstruction_figure_paths"] = hierarchical_reconstruction_figure_paths
manifest["hierarchical_ward_reconstruction_panel_paths"] = hierarchical_reconstruction_panel_paths
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)


Hierarchical Ward-RLN reconstruction data directory: data\processed\02_iceemdan_hierarchical_ward\visualizations\hierarchical_ward_reconstruction_data
Hierarchical Ward-RLN reconstruction curve directory: data\processed\02_iceemdan_hierarchical_ward\visualizations\hierarchical_ward_reconstruction_curves


,series,n_obs,max_abs_error,mean_abs_error,rmse
0,gold_futures,3100,1.776357e-15,8.736529e-17,1.709111e-16
1,silver_futures,3100,7.105427e-15,1.177644e-16,3.000819e-16
2,platinum_futures,3100,3.552714e-15,1.330597e-16,2.652193e-16
3,palladium_futures,3100,5.329071e-15,1.566719e-16,3.476408e-16
4,gold_spot,3100,3.552714e-15,7.807392e-17,1.657013e-16
5,silver_spot,3100,7.105427e-15,1.294512e-16,3.205593e-16
6,platinum_spot,3100,3.552714e-15,1.204490e-16,2.366800e-16
7,palladium_spot,3100,3.552714e-15,1.554921e-16,3.018646e-16
